# Load SinLlama for Embeddings using Unsloth (Memory Optimized)

This notebook demonstrates how to load the `polyglots/SinLlama_v01` adapter with the `meta-llama/Meta-Llama-3-8B` base model using `unsloth` in Google Colab.
**Optimizations**: Reduced sequence length and explicit memory cleaning.

In [40]:
# from google.colab import runtime
# runtime.unassign()

In [41]:
# %%capture
import torch
major_version, minor_version = torch.cuda.get_device_capability()

# Must install unsloth on Colab first
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-69yy5wj6/unsloth_0f0e2b60c16c4bdf9f3c6308659730cb
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-69yy5wj6/unsloth_0f0e2b60c16c4bdf9f3c6308659730cb
  Resolved https://github.com/unslothai/unsloth.git to commit ec1757c1a02175851146ff5f6ab2a26c8c863fc8
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
from unsloth import FastLanguageModel
import torch
import gc
from transformers import AutoTokenizer
from peft import PeftModel

# 1. Clean up before loading
gc.collect()
torch.cuda.empty_cache()

max_seq_length = 1024
dtype = None
load_in_4bit = True
model_config = {"model_name": "unsloth/llama-3-8b", "load_in_4bit": False}

In [ ]:
# Configuration constants
max_seq_length = 1024
dtype = None
load_in_4bit = True

# Define Models Explicitly
base_model_name = "meta-llama/Meta-Llama-3-8B"
adapter_name = "polyglots/SinLlama_v01"

print("STEP 1: Loading Base Model...")
base_model, _ = FastLanguageModel.from_pretrained(
    model_name = base_model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    gpu_memory_utilization = 0.80,
)

print("STEP 2: Resizing Vocabulary Manually...")
base_model.resize_token_embeddings(139336)

print("STEP 3: Loading Extended Tokenizer...")
# 3. Load the tokenizer to check consistency
tokenizer = AutoTokenizer.from_pretrained("polyglots/Extended-Sinhala-LLaMA")
print(f"Tokenizer Len: {len(tokenizer)}, Model Vocab: {base_model.get_input_embeddings().weight.shape[0]}")

print("STEP 4: Loading Adapter...")
# 4. Load the Adapter on top
model = PeftModel.from_pretrained(base_model, adapter_name)

print("Model, Adapter, and Tokenizer loaded successfully!")

STEP 1: Loading Base Model...
==((====))==  Unsloth 2026.1.2: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
STEP 2: Resizing Vocabulary Manually...
STEP 3: Loading Extended Tokenizer...
Tokenizer Len: 139336, Model Vocab: 139336
STEP 4: Loading Adapter...


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1222: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


Model, Adapter, and Tokenizer loaded successfully!


In [44]:
# Sample Sinhala Text
text = "මේක සිංහල භාෂාවෙන් ලියවුනු වාක්‍යයක්."
print(f"Input: {text}")

# Tokenize
inputs = tokenizer(text, return_tensors="pt").to("cuda")

# Generate Embeddings
FastLanguageModel.for_inference(model)

with torch.no_grad():
    outputs = model(**inputs, output_hidden_states=True)

last_hidden_state = outputs.hidden_states[-1]
print(f"Embeddings Shape: {last_hidden_state.shape}")

Input: මේක සිංහල භාෂාවෙන් ලියවුනු වාක්‍යයක්.
Embeddings Shape: torch.Size([1, 9, 4096])


In [45]:
def get_word_level_embeddings(sentence):
    # 1. Tokenize and get word_ids
    inputs = tokenizer(sentence, return_tensors="pt", add_special_tokens=True).to("cuda")
    word_ids = inputs.word_ids()

    # 2. Get the raw token embeddings from the model
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        # Shape: [batch_size, sequence_length, hidden_size]
        # We take [0] to remove the batch dimension since batch_size=1
        token_embeddings = outputs.hidden_states[-1][0]

    # 3. Group sub-tokens by word_id
    word_to_vectors = {}
    word_to_token_ids = {}

    input_ids = inputs["input_ids"][0]

    for i, word_id in enumerate(word_ids):
        if word_id is None: # Skip special tokens like <s>
            continue

        if word_id not in word_to_vectors:
            word_to_vectors[word_id] = []
            word_to_token_ids[word_id] = []

        word_to_vectors[word_id].append(token_embeddings[i])
        word_to_token_ids[word_id].append(input_ids[i])

    # 4. Average vectors and map to decoded words
    final_embeddings = {}

    # We iterate by sorted word_id to maintain order if needed, or just dict order
    for word_id in sorted(word_to_vectors.keys()):
        # Average the vectors for sub-words
        vectors = torch.stack(word_to_vectors[word_id])
        avg_vec = vectors.mean(dim=0)

        # Decode the word from its tokens to get the EXACT text the model saw
        # This is safer than sentence.split() which might not match tokenization (e.g. punctuation)
        word_text = tokenizer.decode(word_to_token_ids[word_id]).strip()

        if word_text:
            final_embeddings[word_text] = avg_vec

    return final_embeddings

In [46]:
embeddings = get_word_level_embeddings(text)

In [47]:
for word, vector in embeddings.items():
    print(f"Word: {word}, Vector Shape: {vector.shape}")

Word: මේක, Vector Shape: torch.Size([4096])
Word: සිංහල, Vector Shape: torch.Size([4096])
Word: භාෂාවෙන්, Vector Shape: torch.Size([4096])
Word: ලිය, Vector Shape: torch.Size([4096])
Word: වුනු, Vector Shape: torch.Size([4096])
Word: වාක්‍ය, Vector Shape: torch.Size([4096])
Word: යක්, Vector Shape: torch.Size([4096])
Word: ., Vector Shape: torch.Size([4096])


In [48]:
import json
from collections import defaultdict
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

In [49]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [50]:
with open("/content/drive/MyDrive/LSD-Analysis/Data/century_tokens.json", "r", encoding="utf-8") as f:
    century_tokens_loaded = json.load(f)

In [51]:
century_embeddings = defaultdict(list)

In [52]:
def get_word_vecs(word_list, max_len=510, window_step=300):
    # 0. Sanitize Input & Handle Strings
    if isinstance(word_list, str):
        # Automatically split strings into tokens
        word_list = word_list.split()

    if not isinstance(word_list, list):
        return []

    # Ensure all elements are strings and remove None/empty
    clean_word_list = [str(w).strip() for w in word_list if w is not None and str(w).strip()]

    if not clean_word_list:
        return []

    # 1. Tokenize using is_split_into_words=True
    try:
        full_inputs = tokenizer(clean_word_list, is_split_into_words=True, return_tensors="pt", add_special_tokens=False)
    except Exception as e:
        print(f"Tokenization failed for input head: {clean_word_list[:5]}... Error: {e}")
        return []

    input_ids = full_inputs["input_ids"][0]
    total_tokens = len(input_ids)

    # Storage for all token-level embeddings
    all_token_embeddings = torch.zeros((total_tokens, model.config.hidden_size))
    processed_counts = torch.zeros(total_tokens)

    # Slide the window across the token IDs
    for start_idx in range(0, total_tokens, window_step):
        end_idx = min(start_idx + max_len, total_tokens)

        # Prepare chunk with special tokens <s> and </s>
        chunk_ids = input_ids[start_idx:end_idx]
        chunk_with_special = torch.cat([
            torch.tensor([tokenizer.bos_token_id]),
            chunk_ids,
            torch.tensor([tokenizer.eos_token_id])
        ]).unsqueeze(0)

        # Move to GPU
        chunk_with_special = chunk_with_special.to(model.device)

        with torch.no_grad():
            # CausalLM returns hidden_states if requested
            outputs = model(chunk_with_special, output_hidden_states=True)
            # Get last layer hidden states: [batch, seq, hidden]
            chunk_embeddings = outputs.hidden_states[-1][0, 1:-1, :]

        # Add to global storage (move back to CPU)
        all_token_embeddings[start_idx:end_idx] += chunk_embeddings.cpu()
        processed_counts[start_idx:end_idx] += 1

    # Average overlapping regions
    final_token_embeddings = all_token_embeddings / processed_counts.unsqueeze(1)

    # Map back to Word IDs
    word_ids = full_inputs.word_ids()
    word_vecs = []
    temp_storage = defaultdict(list)

    for i, word_id in enumerate(word_ids):
        if word_id is not None:
            temp_storage[word_id].append(final_token_embeddings[i])

    # Reconstruct in order of original word_list (using safe indexing)
    for word_idx in range(len(clean_word_list)):
        if word_idx in temp_storage:
            word_vec = torch.stack(temp_storage[word_idx]).mean(dim=0)
            word_vecs.append(word_vec)
        else:
            # Fallback for empty/special words using zeros
            word_vecs.append(torch.zeros(model.config.hidden_size))

    return word_vecs

In [53]:
for century, sentences in century_tokens_loaded.items():
    print(f"Processing century: {century}")

    # Added tqdm for progress tracking
    for sentence_item in tqdm(sentences, desc=f"Century {century}", unit="sent"):
        try:
            # Get vectors (returns list of Tensors)
            vectors = get_word_vecs(sentence_item)

            if not vectors: # Skip if result is empty
                continue

            # Post-process input words for storage
            if isinstance(sentence_item, str):
                 clean_sent_list = sentence_item.split()
            elif isinstance(sentence_item, list):
                 clean_sent_list = [str(w).strip() for w in sentence_item if w is not None and str(w).strip()]
            else:
                continue

            # Accumulate vectors DIRECTLY (Keep as Tensors, no .tolist needed for torch.save)
            # We DO move to CPU to save GPU VRAM, but keep as Tensor
            sentence_export = []
            for word, vec in zip(clean_sent_list, vectors):
                sentence_export.append([word, vec.cpu()]) # Ensure on CPU

            # Accumulate in the global dictionary
            century_embeddings[century].append(sentence_export)

        except Exception as e:
            print(f"Error processing sentence: {e}")

Processing century: 5


Century 5:   0%|          | 0/33 [00:00<?, ?sent/s]

Processing century: 12


Century 12:   0%|          | 0/59 [00:00<?, ?sent/s]

Processing century: 13


...

Century 18:   0%|          | 0/112 [00:00<?, ?sent/s]

Processing century: 19


Century 19:   0%|          | 0/420 [00:00<?, ?sent/s]

Processing century: 20


Century 20:   0%|          | 0/822 [00:00<?, ?sent/s]

In [54]:
word, vector = century_embeddings["5"][0][0]
print(f"\nExample: '{word}' in century 5 has vector of shape {vector.shape}")


Example: 'සම්යක්' in century 5 has vector of shape torch.Size([4096])


In [55]:
century_embeddings["5"]

[[['සම්යක්',
   tensor([ 0.1699,  2.4844, -0.4473,  ..., -0.1953, -0.1602, -0.4102])],
  ['යමු',
   tensor([ 1.9727,  2.5547, -1.1396,  ...,  1.6406,  0.6131, -0.5352])],
  ['බුද්ධාය',
   tensor([-0.5757,  0.1328,  2.2656,  ...,  0.9495,  1.1328,  3.4219])]],
 [['කාම',
   tensor([-1.0889, -2.6953, -0.1934,  ..., -0.2656,  0.3457,  0.6865])],
  ['ලොක',
   tensor([-0.7578,  0.6562,  1.2402,  ...,  2.8125, -0.4824, -0.6367])],
  ['රූප',
   tensor([ 0.4219, -0.1553,  2.1094,  ...,  1.7188,  4.4375,  6.4062])],
  ['ලොක',
   tensor([-0.3984,  0.1680,  2.2734,  ...,  2.3789,  0.7090,  1.0020])],
  ['අරූපලෝක',
   tensor([-0.1156,  0.5664, -2.2904,  ...,  0.2318,  1.4349,  2.8385])],
  ['යන්',
   tensor([-1.1641,  0.5703,  1.4062,  ..., -0.6641, -1.1328,  2.1562])],
  ['ලොකත්රයට',
   tensor([-1.4146,  0.4551,  0.3167,  ..., -1.1669,  0.8105,  0.7500])],
  ['ශ්රෙෂ්ඨ',
   tensor([-1.5850,  1.2695,  0.6979,  ..., -0.7969,  0.7956, -2.1517])],
  ['වූ', tensor([-1.0781,  1.6719, -2.1250,  ...,  1.06

In [59]:
output_file = "/content/drive/MyDrive/LSD-Analysis/Data/SinLlama_All_Century_Embeddings.pt"

In [60]:
torch.save(century_embeddings, output_file)